# Challenge 4: Programming an Agent Workflow
## Self-Verifying Research Assistant (Refines Answers given by AI)

## Use Case
A common problem with AI-generated answers is that the first response is not
always the best one. It may be too brief, miss key details, or lack clarity.
In high-stakes or research-heavy scenarios, you want answers that have been
reviewed and improved before being returned to the user — not just whatever
the model produced on its first attempt.

This notebook builds a Self-Verifying Research Assistant that solves this
problem. When a user asks a question, the system does not immediately return
an answer. Instead, it runs the question through a quality pipeline: search
for an answer, have a critic review and score it, then have a writing
specialist refine it based on the critique. This loop repeats until the answer
scores 7 out of 10 or higher, ensuring the user always receives a well-researched,
clearly written, complete response.

This pattern is especially valuable for complex factual questions — such as
explaining how hurricanes form, summarizing a scientific topic, or researching
a historical event — where depth and accuracy matter.

## How It Works
The **greeter agent** receives the user's question and hands it to the
**answer team** — a loop-based pipeline that iterates until quality is
confirmed. In each iteration, the **search agent** queries Google Search and
saves its findings to shared session state. The **critique agent** reads those
findings, scores the answer out of 10 across accuracy, completeness, clarity,
and length, and saves its feedback to state. The **refine agent** then reads
both the answer and the critique — if the score meets the threshold, it saves
the answer as final and exits the loop. If not, it rewrites the answer
addressing every improvement the critique identified, and the loop runs again
with the improved version.

## Agent Hierarchy
```
greeter  ← receives the user's question, presents the final answer
└── answer_team (LoopAgent, max 2 iterations)
      ├── search_agent   ← researches the question, saves findings to state
      ├── critique_agent ← scores the answer 1-10, saves feedback to state
      └── refine_agent   ← improves the answer or exits if quality passes
```

## Key ADK Concepts Demonstrated
- `LoopAgent`: runs a pipeline repeatedly until an exit condition is met
- `max_iterations`: prevents infinite loops with a hard cap
- Session state: shared memory that lets agents pass data between steps
- Iterative refinement: each loop produces a measurably better answer
- Quality gating: the loop only exits when the answer meets the score threshold

In [1]:
# Dependencies
from google.colab import userdata
import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio
# For Callbacks Logging and Moderation
import logging
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
import subprocess
# Runner and Session
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
# Google Search Tool
from google.adk.tools import google_search
# Agent workflow patterns
from google.adk.agents import SequentialAgent, LoopAgent
print("✅ Dependencies import complete")

✅ Dependencies import complete


In [ ]:
# API Keys
os.environ["GOOGLE_API_KEY"] = "removed for security"
os.environ["GOOGLE_MAPS_API_KEY"] = "removed for security"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

MAPS_API_KEY = os.environ["GOOGLE_MAPS_API_KEY"]

print("✅ Environment variable setup complete")

✅ Environment variable setup complete


In [3]:
# ── Logger Setup ───────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("ch4_workflow")

# Suppress noisy ADK internal logs, keep only our logger + warnings
logging.getLogger("google.adk").setLevel(logging.WARNING)
logging.getLogger("google.genai").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Changed to a valid model version to ensure better quota management
MODEL = "gemini-2.5-flash-lite"

logger.info("✅ Setup complete — model: %s", MODEL)

In [4]:
# Tools: Get Lat Long and Extended Weather

def get_lat_lon(city: str, state: str) -> dict:
    """
    Converts a US city and state into latitude and longitude
    using the Google Maps Geocoding API.

    Args:
        city: Name of the city (e.g. 'Austin').
        state: Two-letter state abbreviation (e.g. 'TX').

    Returns:
        Dictionary with latitude and longitude, or an error message.
    """
    try:
        api_key = MAPS_API_KEY
        params = {"address": f"{city}, {state}, USA", "key": api_key}
        response = requests.get(
            "https://maps.googleapis.com/maps/api/geocode/json",
            params=params, timeout=10
        )
        data = response.json()
        if data["status"] != "OK":
            return {"status": "error", "message": f"Location not found: {city}, {state}"}
        loc = data["results"][0]["geometry"]["location"]
        return {"status": "success", "city": city, "state": state,
                "latitude": loc["lat"], "longitude": loc["lng"]}
    except Exception as e:
        return {"status": "error", "message": str(e)}


def get_extended_weather_forecast(latitude: float, longitude: float) -> dict:
    """
    Retrieves weather forecast and active alerts for a location
    using the National Weather Service API. No API key required.

    Args:
        latitude: Latitude of the location as a float.
        longitude: Longitude of the location as a float.

    Returns:
        Dictionary with forecast periods and active weather alerts.
    """
    try:
        headers = {"User-Agent": "WeatherAgent/1.0 (test@example.com)"}
        points = requests.get(
            f"https://api.weather.gov/points/{latitude},{longitude}",
            headers=headers, timeout=10
        ).json()
        forecast_url = points["properties"]["forecast"]
        periods = requests.get(forecast_url, headers=headers, timeout=10
                               ).json()["properties"]["periods"][:3]
        zone = points["properties"]["forecastZone"].split("/")[-1]
        alerts = requests.get(
            f"https://api.weather.gov/alerts/active/zone/{zone}",
            headers=headers, timeout=10
        ).json().get("features", [])

        return {
            "status": "success",
            "forecast": [{"period": p["name"],
                          "temperature": f"{p['temperature']}°{p['temperatureUnit']}",
                          "forecast": p["shortForecast"]} for p in periods],
            "alerts": [{"event": a["properties"]["event"],
                        "severity": a["properties"]["severity"],
                        "headline": a["properties"]["headline"]}
                       for a in alerts] or ["No active alerts"],
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


In [5]:
# ── State Management Tool ─────────────────────────────────────────────────────────────
# tool_context.state is a dict that ALL agents in the session can read/write
# Session state is a shared key-value dictionary that all agents in the
# same session can read and write. It's how agents pass data to each other

def append_to_state(tool_context, field: str, response: str) -> dict:
    """
    Appends a value to a list stored in session state.
    Used by agents to accumulate results across loop iterations.

    Args:
        tool_context: ADK context object providing access to session state.
        field: The state key to append to (e.g. 'search_results').
        response: The string value to append to the list.

    Returns:
        Dictionary confirming the operation and total entry count.
    """
    existing = tool_context.state.get(field, [])
    tool_context.state[field] = existing + [response]
    count = len(existing) + 1
    logger.info("STATE | append_to_state | field='%s' | total entries: %d", field, count)
    return {"status": "success", "field": field, "total_entries": count}


def set_state(tool_context, field: str, value: str) -> dict:
    """
    Sets a single value in session state, overwriting any existing value.
    Used for scalar values like scores, flags, and final answers.

    Args:
        tool_context: ADK context object providing access to session state.
        field: The state key to set.
        value: The string value to store.

    Returns:
        Dictionary confirming the operation.
    """
    tool_context.state[field] = value
    preview = value[:80].replace("\n", " ") if value else ""
    logger.info("STATE | set_state | field='%s' | value='%s...'", field, preview)
    return {"status": "success", "field": field, "value": value}


logger.info("✅ State management tools defined")

print("✅ State management tools ready")

✅ State management tools ready


In [6]:
# ── Search Agent ───────────────────────────────────────────────────────────
# output_key="search_results" tells ADK to automatically save this
# agent's final response to session state — no append_to_state needed.
# This avoids the built-in tool + custom tool conflict entirely.

search_workflow_agent = Agent(
    name="search_workflow_agent",
    model=MODEL,
    description="Specialist research agent that searches for information to answer user questions.",
    instruction="""
    You are a thorough research agent. Your job is to find high-quality
    information to answer the user's question.

    Follow these steps:
    1. Use google_search to find accurate, up-to-date information on the topic.
    2. If the first search does not give enough detail, search again with
       a more specific query.
    3. Compile your findings into a clear, well-structured answer that:
       - Directly addresses the question
       - Includes key facts, figures, and context
       - Is at least 3-4 paragraphs long
       - Cites the type of sources used (e.g. scientific, government, news)

    Return your compiled answer as your final response.
    Do NOT call any state tools — your output is saved automatically.
    """,
    tools=[google_search],       # ✅ Only built-in tool — no mixing
    output_key="search_results", # ✅ ADK auto-saves response to state
)

logger.info("✅ search_workflow_agent defined — output_key='search_results'")

In [7]:
# ── Critique Agent ─────────────────────────────────────────────────────────
# Reads search_results from state via {search_results} template injection.
# Writes its critique using set_state (custom tool only — no conflict).

critique_agent = Agent(
    name="critique_agent",
    model=MODEL,
    description="Quality reviewer that evaluates research answers and provides scores.",
    instruction="""
    You are a rigorous quality reviewer. Evaluate the research answer below
    and decide if it is good enough to return to the user.

    RESEARCH ANSWER TO REVIEW:
    {search_results}

    Score it on these four criteria:
    - Accuracy (0-3 pts): Are the facts correct and well-supported?
    - Completeness (0-3 pts): Does it fully answer the question?
    - Clarity (0-2 pts): Is it easy to read and well-structured?
    - Length (0-2 pts): Is it detailed enough but not padded?

    Add the scores for a total out of 10.

    Save your critique using set_state with field='critique' in this format:
    "SCORE: X/10 | IMPROVEMENTS: [list specific improvements needed]"

    If score is 7 or above, ALSO call set_state with:
    field='exit_loop' and value='true'
    """,
    tools=[set_state],           # ✅ Only custom tools — no conflict
    output_key="critique",       # ✅ Also auto-saves output to state
)

logger.info("✅ critique_agent defined — output_key='critique'")

In [8]:
# ── Refine Agent ───────────────────────────────────────────────────────────
# Reads both search_results and critique from state via template injection.
# Writes final_answer using set_state only — no built-in tools needed.

refine_agent = Agent(
    name="refine_agent",
    model=MODEL,
    description="Writing specialist that rewrites answers based on critique feedback.",
    instruction="""
    You are a skilled writer and editor. Improve the research answer below
    based on the critique provided.

    CURRENT ANSWER:
    {search_results}

    CRITIQUE:
    {critique}

    Instructions:
    1. Extract the score from the critique (format: "SCORE: X/10 | ...").

    If score is 7 or above:
    - Copy the CURRENT ANSWER exactly as-is into set_state with field='final_answer'
    - Do not summarize or comment — save the full original answer text.

    If score is below 7:
    - Rewrite the answer addressing EVERY improvement in the critique.
    - Your rewrite must be noticeably better than the original.
    - Save using set_state with field='final_answer'

    Always call set_state with field='final_answer' before finishing.
    """,
    tools=[set_state],           # ✅ Only custom tools — no conflict
    output_key="final_answer",   # ✅ Also auto-saves output to state
)

logger.info("✅ refine_agent defined — output_key='final_answer'")

In [9]:
# ── Workflow Assembly (FIXED)
# Re-defining everything to clear Pydantic parent-assignment state
answer_team = LoopAgent(
    name="answer_team",
    description="Iterative research pipeline.",
    sub_agents=[search_workflow_agent, critique_agent, refine_agent],
    max_iterations=2,
)

greeter = Agent(
    name="greeter",
    model=MODEL,
    description="Main assistant.",
    instruction="""
    Greet the user and delegate to the answer_team.
    Present the 'FINAL_RESEARCH_ANSWER' from history at the end.
    """,
    sub_agents=[answer_team],
    tools=[],
)

logger.info("✅ Workflow re-assembled with zero conflicts")

In [10]:
async def ask_workflow_agent(question: str, session_id: str) -> None:
    """
    Runs a question through the full workflow and prints:
    - Live agent activity log showing delegation and tool calls
    - The final answer
    - Session state dump showing data flow between agents
    """
    session_service = InMemorySessionService()
    runner = Runner(
        agent=greeter,
        app_name="ch4_workflow_app",
        session_service=session_service,
    )
    session = await session_service.create_session(
        app_name="ch4_workflow_app",
        user_id="user_001",
        session_id=session_id,
    )

    logger.info("=" * 60)
    logger.info("NEW QUERY | session_id=%s", session_id)
    logger.info("QUESTION: %s", question)
    logger.info("=" * 60)

    print(f"\n{'='*60}")
    print(f"❓ QUESTION: {question}")
    print(f"{'='*60}")
    print("\n📡 LIVE AGENT ACTIVITY:")
    print("-" * 60)

    final_response = ""
    event_count = 0

    async for event in runner.run_async(
        user_id="user_001",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=question)]
        ),
    ):
        event_count += 1
        author = getattr(event, "author", "unknown")

        if hasattr(event, "content") and event.content and event.content.parts:
            for part in event.content.parts:

                # ── Tool calls ─────────────────────────────────────────────
                if hasattr(part, "function_call") and part.function_call:
                    fn_name = part.function_call.name
                    args = dict(part.function_call.args) if part.function_call.args else {}

                    # Summarize args for readability
                    arg_preview = ""
                    if "field" in args:
                        arg_preview = f"field='{args['field']}'"
                    elif "query" in args:
                        arg_preview = f"query='{str(args.get('query',''))[:50]}'"

                    print(f"  🔧 [{author}] → tool: {fn_name}({arg_preview})")
                    logger.info("TOOL CALL | agent=%s | tool=%s | args=%s",
                                author, fn_name, arg_preview)

                # ── Text outputs ───────────────────────────────────────────
                elif hasattr(part, "text") and part.text:
                    preview = part.text.strip()[:120].replace("\n", " ")
                    print(f"  💬 [{author}] → {preview}...")
                    logger.info("TEXT OUTPUT | agent=%s | preview=%s...", author, preview[:80])

        # ── Final response ─────────────────────────────────────────────────
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if hasattr(part, "text") and part.text:
                        final_response = part.text
                        break

    print("-" * 60)
    logger.info("WORKFLOW COMPLETE | events processed: %d", event_count)

    # ── Final Answer ───────────────────────────────────────────────────────
    print(f"\n✅ FINAL ANSWER:")
    print(final_response)

    # ── Session State Dump ─────────────────────────────────────────────────
    # Shows how data flowed between agents — key for demonstrating the workflow
    final_session = await session_service.get_session(
        app_name="ch4_workflow_app",
        user_id="user_001",
        session_id=session_id,
    )

    print(f"\n📦 SESSION STATE (agent data flow):")
    print("-" * 60)
    if final_session and final_session.state:
        for key, val in final_session.state.items():
            if isinstance(val, list):
                print(f"  {key}: [{len(val)} entries]")
                for i, entry in enumerate(val):
                    preview = str(entry)[:100].replace("\n", " ")
                    print(f"    [{i+1}] {preview}...")
            else:
                preview = str(val)[:150].replace("\n", " ")
                print(f"  {key}: {preview}")
            logger.info("STATE DUMP | key=%s | type=%s", key,
                        "list" if isinstance(val, list) else "str")
    else:
        print("  (no state found)")
    print("=" * 60)


logger.info("✅ Runner and test function ready")
print("✅ Runner and test function ready")

✅ Runner and test function ready


In [11]:
async def run_with_retry(runner, user_id, session_id, message, max_retries=3):
    """
    Wraps runner.run_async with retry logic for 429 rate limit errors.
    Collects all events and returns them as a list.
    """
    for attempt in range(max_retries):
        try:
            events = []
            async for event in runner.run_async(
                user_id=user_id,
                session_id=session_id,
                new_message=message,
            ):
                events.append(event)
            return events  # ✅ Success

        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait_time = 60 * (attempt + 1)  # 60s, 120s, 180s
                print(f"\n⏳ Rate limit hit (attempt {attempt+1}/{max_retries}) — waiting {wait_time}s...")
                logger.warning("Rate limit hit attempt %d — waiting %ds", attempt + 1, wait_time)
                await asyncio.sleep(wait_time)
            else:
                raise  # Not a rate limit error — re-raise immediately

    raise Exception("Max retries exceeded — quota still exhausted")


async def ask_workflow_agent(question: str, session_id: str) -> None:
    """
    Runs a question through the full workflow with retry logic.
    """
    session_service = InMemorySessionService()
    runner = Runner(
        agent=greeter,
        app_name="ch4_workflow_app",
        session_service=session_service,
    )
    session = await session_service.create_session(
        app_name="ch4_workflow_app",
        user_id="user_001",
        session_id=session_id,
    )

    logger.info("=" * 60)
    logger.info("NEW QUERY | session_id=%s", session_id)
    logger.info("QUESTION: %s", question)
    logger.info("=" * 60)

    print(f"\n{'='*60}")
    print(f"❓ QUESTION: {question}")
    print(f"{'='*60}")
    print("\n📡 LIVE AGENT ACTIVITY:")
    print("-" * 60)

    message = types.Content(
        role="user",
        parts=[types.Part(text=question)]
    )

    # ✅ All retry logic lives here — outside the async generator
    events = await run_with_retry(
        runner=runner,
        user_id="user_001",
        session_id=session.id,
        message=message,
    )

    final_response = ""

    for event in events:
        author = getattr(event, "author", "unknown")

        if hasattr(event, "content") and event.content and event.content.parts:
            for part in event.content.parts:

                if hasattr(part, "function_call") and part.function_call:
                    fn_name = part.function_call.name
                    args = dict(part.function_call.args) if part.function_call.args else {}
                    arg_preview = ""
                    if "field" in args:
                        arg_preview = f"field='{args['field']}'"
                    elif "query" in args:
                        arg_preview = f"query='{str(args.get('query', ''))[:50]}'"
                    print(f"  🔧 [{author}] → tool: {fn_name}({arg_preview})")
                    logger.info("TOOL CALL | agent=%s | tool=%s | args=%s",
                                author, fn_name, arg_preview)

                elif hasattr(part, "text") and part.text:
                    preview = part.text.strip()[:120].replace("\n", " ")
                    print(f"  💬 [{author}] → {preview}...")
                    logger.info("TEXT OUTPUT | agent=%s | preview=%s...",
                                author, preview[:80])

        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if hasattr(part, "text") and part.text:
                        final_response = part.text
                        break

    print("-" * 60)
    print(f"\n✅ FINAL ANSWER:")
    print(final_response)

    # Session state dump
    final_session = await session_service.get_session(
        app_name="ch4_workflow_app",
        user_id="user_001",
        session_id=session_id,
    )

    print(f"\n📦 SESSION STATE (agent data flow):")
    print("-" * 60)
    if final_session and final_session.state:
        for key, val in final_session.state.items():
            if isinstance(val, list):
                print(f"  {key}: [{len(val)} entries]")
                for i, entry in enumerate(val):
                    preview = str(entry)[:100].replace("\n", " ")
                    print(f"    [{i+1}] {preview}...")
            else:
                preview = str(val)[:150].replace("\n", " ")
                print(f"  {key}: {preview}")
    else:
        print("  (no state found)")
    print("=" * 60)

logger.info("✅ ask_workflow_agent with retry ready")

In [12]:
# # Wait for quota to reset first
# print("⏳ Waiting 60s for quota reset...")
# await asyncio.sleep(60)

# Run single question test
await ask_workflow_agent(
    question="What are the main causes of hurricanes and how do they form?",
    session_id="ch4_session_0"
)


❓ QUESTION: What are the main causes of hurricanes and how do they form?

📡 LIVE AGENT ACTIVITY:
------------------------------------------------------------


  💬 [greeter] → Hello! I'll help you with that....
  🔧 [greeter] → tool: transfer_to_agent()
  💬 [search_workflow_agent] → Hurricanes, also known as typhoons or tropical cyclones depending on their location, are powerful weather phenomena that...
  💬 [critique_agent] → SCORE: 9/10 | IMPROVEMENTS: The answer is very comprehensive and accurate. It clearly explains the formation process of ...
  🔧 [critique_agent] → tool: set_state(field='critique')
  🔧 [critique_agent] → tool: set_state(field='exit_loop')
  🔧 [refine_agent] → tool: set_state(field='final_answer')
  💬 [search_workflow_agent] → Hurricanes, also known as typhoons or tropical cyclones depending on their location, are powerful weather phenomena that...
  💬 [critique_agent] → For context:[refine_agent] said: The answer seems to be well-written and provides a comprehensive explanation of hurrica...
  💬 [critique_agent] → SCORE: 10/10 | IMPROVEMENTS: None...
  💬 [refine_agent] → The provided answer is excellent and requires no i

In [13]:
# Run Comprehensive test
async def run_ch4_tests():
    """
    Runs multiple test scenarios through the self-verifying research workflow.
    Demonstrates the full pipeline across different question types.
    """
    test_scenarios = [
        {
            "label": "SCIENCE",
            "question": "What are the main causes of hurricanes and how do they form?",
            "session_id": "ch4_science"
        },
        {
            "label": "SPACE",
            "question": "What is the James Webb Space Telescope and what has it discovered?",
            "session_id": "ch4_space"
        },
        {
            "label": "HISTORY",
            "question": "What was the Manhattan Project and what was its impact?",
            "session_id": "ch4_history"
        },
    ]

    logger.info("Starting Challenge 4 tests — %d scenarios", len(test_scenarios))
    print(f"\n🚀 CHALLENGE 4: SELF-VERIFYING RESEARCH ASSISTANT")
    print(f"Running {len(test_scenarios)} test scenarios...")
    print("=" * 60)

    for i, scenario in enumerate(test_scenarios):
        print(f"\n📋 SCENARIO {i+1}/{len(test_scenarios)}: [{scenario['label']}]")

        await ask_workflow_agent(
            question=scenario["question"],
            session_id=scenario["session_id"]
        )

        if i < len(test_scenarios) - 1:
            wait = 60
            print(f"\n⏳ Waiting {wait}s before next scenario (rate limit buffer)...")
            logger.info("Waiting %ds between scenarios", wait)
            await asyncio.sleep(wait)

    logger.info("✅ All Challenge 4 scenarios complete")
    print("\n✅ ALL SCENARIOS COMPLETE")

# ── Wait for quota then run all scenarios ─────────────────────────────────
# print("⏳ Waiting 60s for quota reset before starting...")
# await asyncio.sleep(60)
await run_ch4_tests()


🚀 CHALLENGE 4: SELF-VERIFYING RESEARCH ASSISTANT
Running 3 test scenarios...

📋 SCENARIO 1/3: [SCIENCE]

❓ QUESTION: What are the main causes of hurricanes and how do they form?

📡 LIVE AGENT ACTIVITY:
------------------------------------------------------------
  💬 [greeter] → Hello! I'll hand this over to the answer team to research the causes and formation of hurricanes for you....
  🔧 [greeter] → tool: transfer_to_agent()
  💬 [search_workflow_agent] → Hurricanes, also known as typhoons or cyclones depending on the region, are powerful rotating storms that form over warm...
  🔧 [critique_agent] → tool: set_state(field='critique')
  🔧 [critique_agent] → tool: set_state(field='exit_loop')
  🔧 [refine_agent] → tool: set_state(field='final_answer')
  💬 [search_workflow_agent] → Hurricanes, also known as typhoons or cyclones depending on the region, are powerful rotating storms that form over warm...
  🔧 [critique_agent] → tool: set_state(field='critique')
  🔧 [critique_agent] → tool: s